# Readme



# Imports

In [8]:
import sys
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
pd.set_option("display.max_colwidth", None)

# find divs (run once)
go through each url and find div class of main text. save div class to new csv file as "../data/media_body_divs.csv"

In [13]:

"""
Read unique_media_urls.csv, fetch one representative article per media outlet,
and detect the CSS selector (div.class) of the main article-body container.

The heuristic: among all <div> elements, pick the one whose *direct* <p>
children contain the most combined text. That div is almost always the
article body. Fall back to id-based selector if the div has no class.
"""


df = pd.read_csv("../data/unique_media_urls.csv")
df

'\nRead unique_media_urls.csv, fetch one representative article per media outlet,\nand detect the CSS selector (div.class) of the main article-body container.\n\nThe heuristic: among all <div> elements, pick the one whose *direct* <p>\nchildren contain the most combined text. That div is almost always the\narticle body. Fall back to id-based selector if the div has no class.\n'

,media_name,url
0,chinanews.com,http://www.chinanews.com/mil/2018/06-30/8552129.shtml
1,xinhuanet.com,http://www.xinhuanet.com/politics/2018-06/30/c_1123059151.htm
2,voc.com.cn,http://hunan.voc.com.cn/article/201806/201806301027556256.html
3,people.com.cn,http://sports.people.com.cn/worldcup2018/n1/2018/0630/c418684-30097237.html
4,ce.cn,http://ce.cn/xwzx/gnsz/gdxw/201806/30/t20180630_29580863.shtml
5,huanqiu.com,http://world.huanqiu.com/exclusive/2018-07/12394543.html
6,lzbs.com.cn,http://www.lzbs.com.cn/ttnews/2018-07/02/content_4398536.htm
7,01ny.cn,http://news.01ny.cn/2018/fazhizongheng_0713/967320.html
8,cnhubei.com,http://sports.cnhubei.com/2018/0716/399532.shtml
9,syd.com.cn,http://finance.syd.com.cn/system/2018/07/30/011670743.shtml


In [36]:
# functions

# mimic request from real browser (instead of python script)
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/125.0 Safari/537.36")
}

def css_format_converter(tag):
    
    classx = tag.get("class")
    if classx:
        return f"{tag.name}." + ".".join(classx)
        
    idx = tag.get("id")
    if idx:
        return f"{tag.name}#" + idx

    # otherwise
    return tag.name


def find_textbody_tag(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.encoding = r.apparent_encoding or r.encoding
    soup = BeautifulSoup(r.text, "html.parser")


    # find tag that has the most number of <p> children == the main body tag.
    best_tag, best_len = None, 0
    for tag in soup.find_all(True):  # True matches any tag
        direct_p = tag.find_all("p", recursive=False)
        if not direct_p:
            continue
        text = "".join(p.get_text(strip=True) for p in direct_p) # join all text in the <p>s for counting
        if len(text) > best_len:
            best_len = len(text)
            best_tag = css_format_converter(tag)

    if best_tag is None:
        return None, 0
        
    return best_tag, best_len

In [46]:
rows = []
for i, r in df.iterrows():
    media, url = r["media_name"], r["url"]
    try:
        textbody_tag, textlen = find_textbody_tag(url)
        rows.append({
            "media_name": media,
            "url": url,
            "body_div": textbody_tag,
            "body_text_len": textlen,
            "error": None,
        })
        
        print(f"OK   {media:20s} -> {textbody_tag}  ({textlen} chars)")
    except Exception as e:
        rows.append({
            "media_name": media,
            "url": url,
            "body_div": None,
            "body_text_len": 0,
            "error": str(e),
        })
        print(f"FAIL {media:20s} -> {e}", file=sys.stderr)
    time.sleep(1)  # be polite

result = pd.DataFrame(rows)
result.to_csv("../data/results.csv", index=False)
result

OK   chinanews.com        -> div.left_zw  (2021 chars)
OK   xinhuanet.com        -> div#p-detail  (5503 chars)
OK   voc.com.cn           -> div#content  (606 chars)
OK   people.com.cn        -> div.rm_txt_con.cf  (2170 chars)


FAIL ce.cn                -> HTTPConnectionPool(host='ce.cn', port=80): Max retries exceeded with url: /xwzx/gnsz/gdxw/201806/30/t20180630_29580863.shtml (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x15ead0af0>, 'Connection to ce.cn timed out. (connect timeout=20)'))


OK   huanqiu.com          -> None  (0 chars)


FAIL lzbs.com.cn          -> HTTPConnectionPool(host='www.lzbs.com.cn', port=80): Max retries exceeded with url: /ttnews/2018-07/02/content_4398536.htm (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x15ead05a0>, 'Connection to www.lzbs.com.cn timed out. (connect timeout=20)'))


OK   01ny.cn              -> div.hide_me.P.H  (175 chars)


...

FAIL jxnews.com.cn        -> HTTPSConnectionPool(host='news.jxnews.com.cn', port=443): Max retries exceeded with url: /system/2019/05/03/017482261.shtml (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_HANDSHAKE_FAILURE] ssl/tls alert handshake failure (_ssl.c:1032)')))


OK   66wz.com             -> div#artibody  (11956 chars)
OK   qlwb.com.cn          -> None  (0 chars)
OK   yunnan.cn            -> div#layer216  (1283 chars)


FAIL nmgnews.com.cn       -> HTTPConnectionPool(host='sports.nmgnews.com.cn', port=80): Max retries exceeded with url: /system/2019/06/03/012719877.shtml (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x15ead05a0>: Failed to resolve 'sports.nmgnews.com.cn' ([Errno 8] nodename nor servname provided, or not known)"))


OK   cnnb.com.cn          -> None  (0 chars)
OK   fynews.net           -> body  (52 chars)
OK   cjn.cn               -> div.footer  (223 chars)
OK   eastday.com          -> None  (0 chars)
OK   81.cn                -> div.container  (5474 chars)


,media_name,url,body_div,body_text_len,error
0,chinanews.com,http://www.chinanews.com/mil/2018/06-30/8552129.shtml,div.left_zw,2021,None
1,xinhuanet.com,http://www.xinhuanet.com/politics/2018-06/30/c_1123059151.htm,div#p-detail,5503,None
2,voc.com.cn,http://hunan.voc.com.cn/article/201806/201806301027556256.html,div#content,606,None
3,people.com.cn,http://sports.people.com.cn/worldcup2018/n1/2018/0630/c418684-30097237.html,div.rm_txt_con.cf,2170,None
4,ce.cn,http://ce.cn/xwzx/gnsz/gdxw/201806/30/t20180630_29580863.shtml,None,0,"HTTPConnectionPool(host='ce.cn', port=80): Max retries exceeded with url: /xwzx/gnsz/gdxw/201806/30/t20180630_29580863.shtml (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x15ead0af0>, 'Connection to ce.cn timed out. (connect timeout=20)'))"
5,huanqiu.com,http://world.huanqiu.com/exclusive/2018-07/12394543.html,None,0,None
6,lzbs.com.cn,http://www.lzbs.com.cn/ttnews/2018-07/02/content_4398536.htm,None,0,"HTTPConnectionPool(host='www.lzbs.com.cn', port=80): Max retries exceeded with url: /ttnews/2018-07/02/content_4398536.htm (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x15ead05a0>, 'Connection to www.lzbs.com.cn timed out. (connect timeout=20)'))"
7,01ny.cn,http://news.01ny.cn/2018/fazhizongheng_0713/967320.html,div.hide_me.P.H,175,None
8,cnhubei.com,http://sports.cnhubei.com/2018/0716/399532.shtml,None,0,"HTTPConnectionPool(host='sports.cnhubei.com', port=80): Max retries exceeded with url: /2018/0716/399532.shtml (Caused by NameResolutionError(""<urllib3.connection.HTTPConnection object at 0x15ead3680>: Failed to resolve 'sports.cnhubei.com' ([Errno 8] nodename nor servname provided, or not known)""))"
9,syd.com.cn,http://finance.syd.com.cn/system/2018/07/30/011670743.shtml,None,0,"HTTPConnectionPool(host='finance.syd.com.cn', port=80): Max retries exceeded with url: /system/2018/07/30/011670743.shtml (Caused by NameResolutionError(""<urllib3.connection.HTTPConnection object at 0x15ead0af0>: Failed to resolve 'finance.syd.com.cn' ([Errno 8] nodename nor servname provided, or not known)""))"


In [53]:
result2 = result[ ~result.body_div.isna()][["media_name", "body_div"]]
result2.to_csv("../data/media_textbody_tags.csv")
result2

,media_name,body_div
0,chinanews.com,div.left_zw
1,xinhuanet.com,div#p-detail
2,voc.com.cn,div#content
3,people.com.cn,div.rm_txt_con.cf
7,01ny.cn,div.hide_me.P.H
10,bjd.com.cn,div.bjd-not-found__con
11,hangzhou.com.cn,td.Dlife_bcolor02
12,lyd.com.cn,div.page-bottom-word
14,china.com.cn,div.articleBody
15,chinadaily.com.cn,div.article


# /////// fix NaNs

some links do not work: find new urls from the same media company

In [5]:
df = pd.read_csv("../data/media_body_divs.csv")
df

,media_name,url,body_div,body_text_len,error
0,chinanews.com,http://www.chinanews.com/mil/2018/06-30/855212...,div.left_zw,2021,NaN
1,xinhuanet.com,http://www.xinhuanet.com/politics/2018-06/30/c...,div#p-detail,5503,NaN
2,voc.com.cn,http://hunan.voc.com.cn/article/201806/2018063...,div#content,606,NaN
3,people.com.cn,http://sports.people.com.cn/worldcup2018/n1/20...,div.rm_txt_con.cf,2170,NaN
4,ce.cn,http://ce.cn/xwzx/gnsz/gdxw/201806/30/t2018063...,NaN,0,"HTTPConnectionPool(host='ce.cn', port=80): Max..."
5,huanqiu.com,http://world.huanqiu.com/exclusive/2018-07/123...,NaN,0,NaN
6,lzbs.com.cn,http://www.lzbs.com.cn/ttnews/2018-07/02/conte...,NaN,0,"HTTPConnectionPool(host='www.lzbs.com.cn', por..."
7,01ny.cn,http://news.01ny.cn/2018/fazhizongheng_0713/96...,div.hide_me.P.H,175,NaN
8,cnhubei.com,http://sports.cnhubei.com/2018/0716/399532.shtml,NaN,0,"HTTPConnectionPool(host='sports.cnhubei.com', ..."
9,syd.com.cn,http://finance.syd.com.cn/system/2018/07/30/01...,NaN,0,"HTTPConnectionPool(host='finance.syd.com.cn', ..."


In [9]:
df.iloc[11]

media_name                                                               hangzhou.com.cn
url              http://jrsh.hangzhou.com.cn/sale/content/2018-07/31/content_7044621.htm
body_div                                                               div (no class/id)
body_text_len                                                                        102
error                                                                                NaN
Name: 11, dtype: object